# Strained CALPHAD model for wurtzite Al$_{1-x}$Sc$_x$N

Phase separation of wurtzite Al$_{1-x}$Sc$_x$N as bulk and as a film on a substrate: binodals, spinodals, critical
temperatures and the decomposition driving force, for incoherent and coherent domains.

Every literature number here was extracted and validated in the companion notebook
`Hirata2020_AlScN_CALPHAD.ipynb`; each input cell names the section it comes from. Energies are J per mol of formula
unit internally and meV per formula unit (f.u. = Al$_{1-x}$Sc$_x$N) in tables and plots.

Navigation: section 2 holds everything adjustable, section 4 the physics, section 6 the results. Sections 1, 3 and 5
are machinery and can stay collapsed.

Sources: K. Hirata *et al.*, *J. Eur. Ceram. Soc.* **40**, 5410 (2020),
[10.1016/j.jeurceramsoc.2020.06.047](https://doi.org/10.1016/j.jeurceramsoc.2020.06.047) (free energy, lattice
parameters, elastic moduli); S. Zhang *et al.*, *J. Appl. Phys.* **114**, 243516 (2013),
[10.1063/1.4848036](https://doi.org/10.1063/1.4848036) and D. Holec, PhD thesis, Cambridge (2008) (critical
thickness).

# 1. Smooth-function machinery

Shared by the interpolated inputs of section 2. Nothing physical here.

In [ ]:
import dataclasses
import functools
import math
import operator

import numpy as np
from scipy.optimize import brentq, minimize, minimize_scalar
from scipy.special import expit, xlogy

try:
    import matplotlib.pyplot as plt
    HAVE_MATPLOTLIB = True
except ImportError:
    HAVE_MATPLOTLIB = False


def softplus(value, width):
    """Smooth max(value, 0) with the given width; accepts arrays."""
    return width * np.logaddexp(0.0, np.asarray(value, dtype=float) / width)


def smoothed_branch_fit(composition, parameters, width):
    """Two-branch fit [p0, p1, p2, p4] with its sqrt and kink at x0 replaced by softplus forms."""
    x = np.asarray(composition, dtype=float)
    above = softplus(x - TRANSITION_COMPOSITION, width)
    return (parameters[0] + parameters[1] * (x - above) + parameters[3] * above
            + parameters[2] * np.sqrt(softplus(TRANSITION_COMPOSITION - x, 0.5 * width)))


def joined_branches(composition, polynomials, width):
    """Buckled and flat polynomial fits blended by a logistic weight centred on x0."""
    x = np.asarray(composition, dtype=float)
    weight = expit((x - TRANSITION_COMPOSITION) / width)
    return (1.0 - weight) * np.polyval(polynomials[0], x) + weight * np.polyval(polynomials[1], x)

# 2. Inputs

Everything adjustable lives here.

## 2.1 Literature data

### 2.1.1 Hirata et al. (2020) — assessed free energy

#### 2.1.1.1 Tabulated

In [ ]:
# Wurtzite (Al,Sc)N Redlich-Kister parameters, J per mol of formula unit. Table 2; the sign of L2 is lost from the PDF
# text and fixed by Fig. 14 - Hirata notebook sections 5 and 6a.
REDLICH_KISTER_WURTZITE = (97970.4838186, 434.322298892, -17960.9058372)    # L0, L1, L2

### 2.1.2 Hirata et al. (2020) — lattice parameters

#### 2.1.2.1 Tabulated

In [ ]:
# Continuity-constrained two-branch fit to the Fig. 11 data, Angstrom - Hirata notebook section 15 (printed in 16c).
# Parameters [p0, p1, p2, p4]: x < x0: p0 + p1 x + p2 sqrt(x0 - x);  x > x0: p0 + p1 x0 + p4 (x - x0).
TRANSITION_COMPOSITION = 0.5555                                      # x0, buckled -> flat
LATTICE_A_FIT = (3.398373741033616, 0.17544525954074075, -0.38563833826503907, 0.47273298890089305)
LATTICE_C_FIT = (3.5607476553295663, 1.5834721310585416, 1.9040612417880973, 0.09698571729425413)

#### 2.1.2.2 Interpolants

In [ ]:
def lattice_parameters(composition):
    """Stress-free (a, c) in Angstrom of a homogeneous alloy, transition smoothed over lattice_transition_width."""
    return (smoothed_branch_fit(composition, LATTICE_A_FIT, lattice_transition_width),
            smoothed_branch_fit(composition, LATTICE_C_FIT, lattice_transition_width))

### 2.1.3 Hirata et al. (2020) — elastic moduli

#### 2.1.3.1 Tabulated

In [ ]:
# Isotropic (Voigt-Reuss-Hill) moduli of Fig. 10: quadratic fit on the buckled branch (x <= 0.5), linear on the flat
# branch (x >= 0.75), highest power first - Hirata notebook section 16a. E within 1.5 %, nu within 0.0016.
YOUNGS_MODULUS_BRANCH_FITS = ((135.04, -409.52, 342.41), (18.32, 159.37333333333305))         # GPa
POISSON_RATIO_BRANCH_FITS = ((-0.0777142857142858, 0.21805714285714273, 0.24555142857142848),
                             (-0.011599999999999593, 0.3536166666666662))

#### 2.1.3.2 Interpolants

In [ ]:
def youngs_modulus(composition):
    """Isotropic Young's modulus, Pa."""
    return joined_branches(composition, YOUNGS_MODULUS_BRANCH_FITS, lattice_transition_width) * 1e9


def poisson_ratio(composition):
    return joined_branches(composition, POISSON_RATIO_BRANCH_FITS, lattice_transition_width)

### 2.1.4 Hirata et al. (2020) — misfit relaxation

In [ ]:
def strain_relaxation_factor(film_thickness_nm, critical_thickness_nm):
    """Fraction of the coherent strain energy kept above h_c: (h_c/h)(1 + ln(h/h_c)). Hirata Eq. (21), section 7."""
    if film_thickness_nm <= critical_thickness_nm:
        return 1.0
    ratio = critical_thickness_nm / film_thickness_nm
    return ratio * (1.0 + math.log(film_thickness_nm / critical_thickness_nm))

### 2.1.5 Substrates

In [ ]:
# In-plane lattice parameter, Angstrom. Experimental; assumed on the same scale as Hirata's a(x), whose a(0) is within
# 0.03 % of experimental AlN. AlN itself is taken from the model's own a(0) in 2.2.4.
SUBSTRATE_IN_PLANE_LATTICE = {"GaN": 3.189}

## 2.2 Model inputs

Not from any paper — chosen here, and meant to be edited. Every assumed value is marked.

### 2.2.1 Free-energy corrections

In [ ]:
# Optional corrections to Hirata's G, off by default: add "short_range_order" and/or "excess_vibrational_entropy".
chemical_corrections = ()
# Short-range order: second-order estimate on a cation sublattice of coordination z. A high-temperature expansion,
# not valid far below T_c.
cation_coordination_number = 12
# Excess vibrational entropy of mixing at x = 0.5, k_B per atom. ASSUMED: typical range 0.1-0.2.
excess_vibrational_entropy_kb_per_atom_at_half = 0.15

### 2.2.2 Assumed shape parameters

In [ ]:
# Width over which the buckled -> flat transition at x0 is smeared in a(x), c(x), E(x), nu(x). ASSUMED; the
# sqrt fit itself has an infinite slope at x0. Controls the strength of every effect near x0.
lattice_transition_width = 0.02
# Smooth floor of the critical thickness at the dislocation core radius, nm. ASSUMED.
core_radius_floor_width_nm = 0.05
# 2*C13/C33 (x) for the strained c axis. None -> isotropic surrogate 2 nu/(1 - nu). Zhang et al. (2013) give it up to
# x = 0.375 (Hirata notebook 18d: the surrogate is ~30 % low at x = 0.375); not used yet.
out_of_plane_coefficient_from_elastic_constants = None

### 2.2.3 Dislocation core parameters

$a+c$-type core energy $E_c(x)$ (eV/Å) and radius $r_c(x)$ (nm) for the critical thickness. **All DUMMIES**: atomistic
values exist only for GaN and for $a$-type cores; scaling GaN $a+c$ by the AlN/GaN $a$-type ratio gives 3.36 eV/Å and
0.92 nm for AlN (Hirata notebook section 19). ScN: no data.

In [ ]:
def dummy_core_energy_high_constant(composition):
    """eV/Angstrom, the AlN estimate at every x. DUMMY."""
    return 3.3 + 0.0 * np.asarray(composition, dtype=float)


def dummy_core_energy_low_constant(composition):
    """eV/Angstrom, 3.1 at every x. DUMMY."""
    return 3.1 + 0.0 * np.asarray(composition, dtype=float)


def dummy_core_energy_high_slope(composition):
    """eV/Angstrom, linear from 3.3 (AlN) to 3.0 (ScN). DUMMY."""
    return 3.3 - 0.3 * np.asarray(composition, dtype=float)


def dummy_core_radius(composition):
    """nm, GaN a+c scaled by the AlN/GaN a-type ratio. DUMMY."""
    return 0.72 * 0.83 / 0.65 + 0.0 * np.asarray(composition, dtype=float)

### 2.2.4 Film and growth conditions

In [ ]:
growth_temperatures = (573.15, 673.15, 723.15)          # K: 300, 400, 450 C
growth_temperature = 673.15
real_film_thickness_nm = 1000.0
# In-plane lattice of each substrate, Angstrom; AlN = the model's own a(0), so the misfit vanishes at x = 0.
substrate_in_plane_lattice = {"AlN": float(lattice_parameters(0.0)[0]), **SUBSTRATE_IN_PLANE_LATTICE}
aluminium_nitride = substrate_in_plane_lattice["AlN"]
chosen_core_energy = dummy_core_energy_high_slope        # used by every thickness-dependent result
chosen_core_radius = dummy_core_radius

### 2.2.5 Numerical settings

In [ ]:
interface_angles = np.linspace(0.0, 0.5 * math.pi, 19)   # coherent laminate: 0 = basal, 90 deg = prism
partner_grid_points = 48            # starting-point grid per axis, spaced finely near the average composition
refinement_starts = 3               # best grid pairs refined, and as many well-separated ones
refinement_method = "L-BFGS-B"      # gradient-based; "Nelder-Mead" is the derivative-free backup
small_split = 2e-3                  # extra starting point x -+ small_split
separation = 0.05                   # minimum x_C - x_B of the well-separated starting pairs
composition_margin = 1e-7
fluctuation_amplitude = 1e-3        # split used for the spinodal curvature
decomposition_threshold = 1e-3      # J/mol-f.u.: binodal = where the driving force first exceeds this
binodal_temperatures = np.arange(300.0, 7400.0, 250.0)
binodal_scan = np.linspace(0.01, 0.99, 33)
spinodal_temperatures = np.arange(300.0, 7400.0, 100.0)
spinodal_compositions = np.linspace(0.005, 0.995, 199)
critical_compositions = np.linspace(0.01, 0.99, 197)

# 3. Physical constants

In [ ]:
GAS_CONSTANT = 8.314462618                 # J/(mol K)
JOULE_PER_ELECTRONVOLT = 96485.33212       # J/mol per eV per particle
MILLIELECTRONVOLT_PER_JOULE_PER_MOLE = 1000.0 / JOULE_PER_ELECTRONVOLT
AVOGADRO_CONSTANT = 6.02214076e23          # 1/mol
ELECTRONVOLT_PER_ANGSTROM_IN_JOULE_PER_METRE = 1.602176634e-19 / 1e-10
CUBIC_METRE_PER_CUBIC_ANGSTROM = 1e-30
METRE_PER_ANGSTROM = 1e-10
METRE_PER_NANOMETRE = 1e-9
FORMULA_UNITS_PER_WURTZITE_CELL = 2
ATOMS_PER_FORMULA_UNIT = 2

# 4. Model

4.1 holds the ingredients every case uses, 4.2 the elastic energy of a decomposed film for each geometry, 4.3 the
decomposition and stability tests at one composition, 4.4 their assembly into phase boundaries.

## 4.1 Shared ingredients

### 4.1.1 Chemical free energy

$G(x,T) = x(1-x)\sum_n L^{(n)}(1-2x)^n - T S_\mathrm{conf}(x)$ relative to the AlN–ScN wurtzite endmembers, ideal mixing
on the cation sublattice, plus the optional corrections of 2.2.1. One function spans both the buckled and flat
branches, as in Hirata.

In [ ]:
def excess_gibbs_energy(composition):
    """Redlich-Kister excess Gibbs energy of wurtzite, J/mol-f.u.; accepts arrays."""
    x = np.asarray(composition, dtype=float)
    return x * (1.0 - x) * sum(coefficient * (1.0 - 2.0 * x) ** order
                               for order, coefficient in enumerate(REDLICH_KISTER_WURTZITE))


def configurational_entropy(composition):
    """Ideal cation-sublattice mixing entropy, J/(mol-f.u. K); accepts arrays."""
    x = np.asarray(composition, dtype=float)
    return -GAS_CONSTANT * (xlogy(x, x) + xlogy(1.0 - x, 1.0 - x))


def short_range_order_correction(composition, temperature):
    """Second-order (Onsager/TAP) SRO free energy, J/mol-f.u."""
    x = np.asarray(composition, dtype=float)
    interaction = REDLICH_KISTER_WURTZITE[0]
    return -interaction ** 2 * x ** 2 * (1.0 - x) ** 2 / (
        cation_coordination_number * GAS_CONSTANT * temperature)


def excess_vibrational_entropy(composition):
    """Guessed sigma*x(1-x) excess vibrational entropy of mixing, J/(mol-f.u. K)."""
    x = np.asarray(composition, dtype=float)
    per_formula_unit_at_half = (excess_vibrational_entropy_kb_per_atom_at_half * GAS_CONSTANT
                                * ATOMS_PER_FORMULA_UNIT)
    return 4.0 * per_formula_unit_at_half * x * (1.0 - x)


def chemical_gibbs_energy_of_mixing(composition, temperature, corrections=()):
    """Wurtzite G_mix(x, T), J/mol-f.u.; corrections: 'short_range_order', 'excess_vibrational_entropy'."""
    x = np.asarray(composition, dtype=float)
    energy = (excess_gibbs_energy(x)
              - temperature * configurational_entropy(x))
    if "short_range_order" in corrections:
        energy = energy + short_range_order_correction(x, temperature)
    if "excess_vibrational_entropy" in corrections:
        energy = energy - temperature * excess_vibrational_entropy(x)
    return energy


wurtzite_gibbs_energy = functools.partial(chemical_gibbs_energy_of_mixing, corrections=chemical_corrections)

### 4.1.2 Elastic properties of a homogeneous film

Isotropic elasticity throughout. A (0001) film clamped in-plane keeps its $c$ free: biaxial modulus $E/(1-\nu)$,
out-of-plane strain $-\,2C_{13}/C_{33}$ times the in-plane one (isotropic surrogate unless 2.2.2 supplies it).

In [ ]:
def molar_volume(composition):
    """m^3 per mol of formula unit."""
    a, c = lattice_parameters(composition)
    cell_volume = np.sqrt(3.0) / 2.0 * a * a * c * CUBIC_METRE_PER_CUBIC_ANGSTROM
    return cell_volume * AVOGADRO_CONSTANT / FORMULA_UNITS_PER_WURTZITE_CELL


def out_of_plane_coefficient(composition):
    """-eps_zz/eps_xx of a (0001) film under biaxial in-plane strain: 2*C13/C33."""
    if out_of_plane_coefficient_from_elastic_constants is not None:
        return out_of_plane_coefficient_from_elastic_constants(composition)
    nu = poisson_ratio(composition)
    return 2.0 * nu / (1.0 - nu)          # isotropic surrogate


def strained_lattice_parameter_c(composition, substrate_lattice_parameter):
    """c (Angstrom) of a (0001) film clamped in-plane to a substrate lattice parameter (Angstrom)."""
    a, c = lattice_parameters(composition)
    in_plane_strain = substrate_lattice_parameter / a - 1.0
    return c * (1.0 - out_of_plane_coefficient(composition) * in_plane_strain)


def plane_stress_stiffness(compositions):
    """In-plane stiffness (Pa) of an isotropic solid with free normal stress; shape (..., 2, 2)."""
    poisson = poisson_ratio(compositions)
    ones = np.ones_like(poisson)
    matrix = np.stack([np.stack([ones, poisson], -1), np.stack([poisson, ones], -1)], -2)
    return (youngs_modulus(compositions) / (1.0 - poisson ** 2))[..., None, None] * matrix


def lame_constants(compositions):
    """(lambda, lambda + 2 mu) of the isotropic solid, Pa."""
    youngs, poisson = youngs_modulus(compositions), poisson_ratio(compositions)
    lame = youngs * poisson / ((1.0 + poisson) * (1.0 - 2.0 * poisson))
    return lame, lame + youngs / (1.0 + poisson)


def isotropic_energy_density(strain_t, strain_n, strain_z, lame, longitudinal):
    """Elastic energy density (J/m^3) for normal strains only."""
    return (0.5 * longitudinal * (strain_t ** 2 + strain_n ** 2 + strain_z ** 2)
            + lame * (strain_t * strain_n + strain_t * strain_z + strain_n * strain_z))

### 4.1.3 Critical thickness and misfit relaxation

Energy balance of Zhang *et al.* (2013) and Holec (2008): a misfit dislocation is favoured once the misfit energy it
relieves exceeds its own energy, $b_\perp M|\epsilon|\,h_c = A\ln(h_c/r_c) + E_c$, with $b_\perp = (\sqrt3/2)a$ for the
$\tfrac13\langle11\bar23\rangle\{1\bar101\}$ slip system, $M = E/(1-\nu)$ and the isotropic line-energy prefactor
$A = \mu(a^2+c^2-\nu a^2/4)/[4\pi(1-\nu)]$. Validated against Zhang's inputs in Hirata notebook 18e, where Zhang's
"2 nm" at $x = 0.375$ is not reproduced (1.2 nm). Above $h_c$ the film keeps the fraction $f(h, h_c)$ (2.1.4) of its
coherent energy; a coherent film keeps the fraction $\sqrt f$ of its misfit.

In [ ]:
def misfit_balance(thickness_nm, released_per_nm, prefactor, core_energy, core_radius_nm):
    """Relieved misfit energy minus dislocation energy per unit length, J/m; zero at h_c."""
    return released_per_nm * thickness_nm - prefactor * math.log(thickness_nm / core_radius_nm) - core_energy


def dislocation_energy_terms(composition, substrate_lattice_parameter):
    """(relieved energy per nm of thickness, line-energy prefactor A), J/m, isotropic elasticity."""
    a, c = (float(value) for value in lattice_parameters(composition))
    youngs, poisson = float(youngs_modulus(composition)), float(poisson_ratio(composition))
    misfit = abs(substrate_lattice_parameter / a - 1.0)
    released = math.sqrt(3.0) / 2.0 * a * METRE_PER_ANGSTROM * youngs / (1.0 - poisson) * misfit * METRE_PER_NANOMETRE
    shear = youngs / (2.0 * (1.0 + poisson))
    prefactor = shear * (a * a + c * c - poisson * a * a / 4.0) * METRE_PER_ANGSTROM ** 2 / (4.0 * math.pi * (1.0 - poisson))
    return released, prefactor


def critical_thickness_nm(composition, core_energy, core_radius,
                          substrate_lattice_parameter=substrate_in_plane_lattice["AlN"]):
    """Equilibrium critical thickness (nm) of a (0001) film; core_energy(x) in eV/Angstrom, core_radius(x) in nm.

    The larger root of the energy balance; where none exists (dislocations favoured at any thickness) the turning
    point of the balance, which keeps h_c continuous. Then smoothly floored at the core radius, below which the
    model does not apply. Infinity for a lattice-matched film.
    """
    released, prefactor = dislocation_energy_terms(composition, substrate_lattice_parameter)
    if released == 0.0:
        return math.inf
    radius = float(core_radius(composition))
    arguments = (released, prefactor, float(core_energy(composition)) * ELECTRONVOLT_PER_ANGSTROM_IN_JOULE_PER_METRE,
                 radius)
    turning_point = prefactor / released                      # minimum of the balance, nm
    thickness = turning_point
    if misfit_balance(turning_point, *arguments) < 0.0:
        upper = 2.0 * turning_point
        while misfit_balance(upper, *arguments) < 0.0:
            upper *= 2.0
        thickness = brentq(misfit_balance, turning_point, upper, args=arguments)
    width = core_radius_floor_width_nm
    return radius + width * float(np.logaddexp(0.0, (thickness - radius) / width))


@functools.lru_cache(maxsize=None)
def inverse_critical_thickness_table(core_energy, core_radius, substrate_lattice_parameter):
    """1/h_c (1/nm) on a composition mesh; 0 where the film never relaxes."""
    mesh = np.linspace(0.0, 1.0, 201)
    return mesh, np.array([1.0 / critical_thickness_nm(x, core_energy, core_radius, substrate_lattice_parameter)
                           for x in mesh])


def relaxation_factors(compositions, film_thickness_nm, core_energy, core_radius, substrate_lattice_parameter):
    """Hirata's f(h) = (h_c/h)(1 + ln(h/h_c)) for h > h_c, else 1, with h_c(x) from the core parameters."""
    mesh, inverse = inverse_critical_thickness_table(core_energy, core_radius, float(substrate_lattice_parameter))
    ratio = film_thickness_nm * np.interp(compositions, mesh, inverse)          # h / h_c
    return np.where(ratio <= 1.0, 1.0, (1.0 + np.log(np.maximum(ratio, 1.0))) / np.maximum(ratio, 1.0))


def partially_relaxed_lattice(composition, substrate_lattice_parameter, film_thickness_nm, core_energy, core_radius):
    """In-plane lattice imposed on a coherent film of average composition x, partly relaxed by dislocations."""
    own = float(lattice_parameters(composition)[0])
    retained = math.sqrt(float(relaxation_factors(np.array(composition), film_thickness_nm, core_energy,
                                                  core_radius, substrate_lattice_parameter)))
    return own * (1.0 + retained * (substrate_lattice_parameter / own - 1.0))

## 4.2 Elastic energy of a decomposed film

A decomposed film consists of domains $i$ with compositions $x_i$, mole fractions $f_i$ and stress-free lattices
$a(x_i)$, $c(x_i)$. Each model returns $\sum_i f_i V_m(x_i)\,w_i$ in J per mol of formula unit of the whole film, for
arrays of shape (..., domains). Domains are flat layers much thicker than the interface; no interface energy.

| model | geometry | constraint |
|---|---|---|
| `IncoherentDomains` | dislocated interfaces | none |
| `CoherentLaminate` | free-standing (bulk, or a film fully relaxed from its substrate) | domains share the lattice in the interface plane; the interface orientation $\theta$ (from $c$) is a variable |
| `ClampedLayered` | film on a substrate, interfaces parallel to it | every layer clamped in-plane to $a_\mathrm{sub}$ |
| `ClampedLateral` | film on a substrate, columns | domains share the lattice along the interface; their average in-plane lattice is clamped |
| `RelaxedClampedDomains` | film on a substrate, partly relaxed; Hirata's film model | each domain clamped and relaxed separately by $f(h, h_c(x_i))$; no strain between domains |

### 4.2.1 Domain bookkeeping and the common interface

In [ ]:
def domain_properties(compositions, fractions):
    """Lattice misfits relative to the average-composition lattice, and fraction * molar volume."""
    compositions = np.asarray(compositions, dtype=float)
    fractions = np.asarray(fractions, dtype=float)
    reference_a, reference_c = lattice_parameters(np.sum(fractions * compositions, axis=-1))
    a, c = lattice_parameters(compositions)
    misfit_a = a / reference_a[..., None] - 1.0
    misfit_c = c / reference_c[..., None] - 1.0
    return misfit_a, misfit_c, fractions * molar_volume(compositions), reference_a


def energy_at_angle(angle, model, compositions, fractions):
    """Scalar elastic energy of one configuration at one interface angle, for 1-D minimisation."""
    return float(np.squeeze(model.energy(compositions, fractions, angle)))


class ElasticModel:
    """Elastic energy (J/mol-f.u.) of a set of domains: compositions, fractions of shape (..., domains) -> (...).

    `angle` is the interface orientation (radians from c) for models that have one; others ignore it.
    """

    def energy(self, compositions, fractions, angle=None):
        raise NotImplementedError

    def energy_and_best_angle(self, compositions, fractions):
        """Energy at the softest orientation on the interface_angles grid, and that angle (None if no orientation)."""
        return self.energy(compositions, fractions), None

    def minimum_energy(self, compositions, fractions):
        """Energy at the softest orientation, found continuously; for a single configuration."""
        return self.energy(compositions, fractions)

### 4.2.2 Incoherent domains

In [ ]:
def no_elastic_energy(compositions, fractions):
    """Incoherent domains: zero, J/mol-f.u."""
    return np.zeros(np.shape(compositions)[:-1])


class IncoherentDomains(ElasticModel):
    def energy(self, compositions, fractions, angle=None):
        return no_elastic_energy(compositions, fractions)

### 4.2.3 Coherent laminate, free-standing

For an interface normal at angle $\theta$ from $c$, the in-plane misfits are $\epsilon_a$ and
$\epsilon_a\cos^2\theta + \epsilon_c\sin^2\theta$; normal and out-of-plane shear stresses relax (plane stress), and the
shared in-plane lattice minimises the total, a $2\times2$ solve. Reduces to Cahn–Larché for constant moduli and linear
misfit.

In [ ]:
def coherent_laminate_energy_by_angle(compositions, fractions, angles=interface_angles):
    """Free-standing coherent laminate, J/mol-f.u., for each interface angle (radians from c): (..., angles)."""
    misfit_a, misfit_c, weights, _ = domain_properties(compositions, fractions)
    tilted = misfit_a[..., None] * np.cos(angles) ** 2 + misfit_c[..., None] * np.sin(angles) ** 2
    misfits = np.stack([np.broadcast_to(misfit_a[..., None], tilted.shape), tilted], axis=-1)
    weighted = weights[..., None, None] * plane_stress_stiffness(compositions)   # (..., domains, 2, 2)
    load = np.einsum("...dij,...daj->...ai", weighted, misfits)                 # (..., angles, 2)
    total = np.broadcast_to(weighted.sum(axis=-3)[..., None, :, :], load.shape + (2,))
    shared = np.linalg.solve(total, load[..., None])[..., 0]
    elastic = shared[..., None, :, :] - misfits                                 # (..., domains, angles, 2)
    return 0.5 * np.einsum("...dai,...dij,...daj->...a", elastic, weighted, elastic)


class CoherentLaminate(ElasticModel):
    """Free-standing coherent laminate; angle=None takes the softest orientation on the angle grid."""

    def energy(self, compositions, fractions, angle=None):
        angles = interface_angles if angle is None else np.array([angle])
        return coherent_laminate_energy_by_angle(compositions, fractions, angles).min(axis=-1)

    def energy_and_best_angle(self, compositions, fractions):
        by_angle = coherent_laminate_energy_by_angle(compositions, fractions)
        return by_angle.min(axis=-1), interface_angles[np.argmin(by_angle, axis=-1)]

    def minimum_energy(self, compositions, fractions):
        start = float(np.squeeze(self.energy_and_best_angle(compositions, fractions)[1]))
        step = interface_angles[1] - interface_angles[0]
        bounds = (max(start - step, 0.0), min(start + step, 0.5 * math.pi))
        result = minimize_scalar(energy_at_angle, bounds=bounds, args=(self, compositions, fractions),
                                 method="bounded", options={"xatol": 1e-4})
        return min(result.fun, float(np.squeeze(self.energy(compositions, fractions, start))))

### 4.2.4 Coherent domains in a clamped film

Layered: each layer biaxially clamped to $a_\mathrm{sub}$. Lateral: the film normal is $c$, the interface normal lies in
the basal plane; along the interface every domain is at $a_\mathrm{sub}$, along $c$ the domains share one strain with
zero average stress, along the interface normal their fraction-weighted strain is fixed by the substrate — a
$4\times4$ Lagrange solve.

In [ ]:
def clamping_energies(compositions, substrate_lattice_parameter):
    """Biaxial clamping energy of each domain on its own, J per mol of formula unit of that domain."""
    compositions = np.asarray(compositions, dtype=float)
    a, _ = lattice_parameters(compositions)
    strain = np.asarray(substrate_lattice_parameter)[..., None] / a - 1.0
    biaxial = youngs_modulus(compositions) / (1.0 - poisson_ratio(compositions))
    return molar_volume(compositions) * biaxial * strain ** 2


def clamped_layered_energy(compositions, fractions, substrate_lattice_parameter):
    """Every domain biaxially clamped to the substrate (basal interfaces); J/mol-f.u."""
    return np.sum(np.asarray(fractions) * clamping_energies(compositions, substrate_lattice_parameter), axis=-1)


def lateral_constrained_system(weights, lame, longitudinal, misfit_a, misfit_c, strain_t, fractions):
    """Lagrange system for (u_B, u_C, w, multiplier): normal strains of the two domains and shared c strain."""
    shape = weights.shape[:-1]
    matrix = np.zeros(shape + (4, 4))
    right = np.zeros(shape + (4,))
    for domain in (0, 1):
        weight, lam, lon = weights[..., domain], lame[..., domain], longitudinal[..., domain]
        elastic_t = strain_t[..., domain]
        matrix[..., domain, domain] = weight * lon
        matrix[..., domain, 2] = matrix[..., 2, domain] = weight * lam
        matrix[..., 2, 2] += weight * lon
        right[..., domain] = -weight * (-lon * misfit_a[..., domain] + lam * (elastic_t - misfit_c[..., domain]))
        right[..., 2] -= weight * (-lon * misfit_c[..., domain] + lam * (elastic_t - misfit_a[..., domain]))
        matrix[..., domain, 3] = matrix[..., 3, domain] = fractions[..., domain]
    return matrix, right


def clamped_lateral_energy(compositions, fractions, substrate_lattice_parameter):
    """Columnar domains on a clamping substrate (interfaces normal to the film); J/mol-f.u.

    compositions, fractions: shape (..., 2); a single domain falls back to the layered form.
    """
    compositions = np.asarray(compositions, dtype=float)
    fractions = np.asarray(fractions, dtype=float)
    if compositions.shape[-1] == 1:
        return clamped_layered_energy(compositions, fractions, substrate_lattice_parameter)
    misfit_a, misfit_c, weights, reference_a = domain_properties(compositions, fractions)
    substrate_strain = np.asarray(substrate_lattice_parameter) / reference_a - 1.0
    strain_t = substrate_strain[..., None] - misfit_a
    lame, longitudinal = lame_constants(compositions)
    matrix, right = lateral_constrained_system(weights, lame, longitudinal, misfit_a, misfit_c,
                                               strain_t, fractions)
    right[..., 3] = substrate_strain
    solution = np.linalg.solve(matrix, right[..., None])[..., 0]
    strain_n = solution[..., :2] - misfit_a
    strain_z = solution[..., 2:3] - misfit_c
    density = isotropic_energy_density(strain_t, strain_n, strain_z, lame, longitudinal)
    return np.sum(weights * density, axis=-1)


class ClampedLayered(ElasticModel):
    """Every domain biaxially clamped to a substrate in-plane lattice parameter (Angstrom)."""

    def __init__(self, substrate_lattice_parameter):
        self.substrate_lattice_parameter = substrate_lattice_parameter

    def energy(self, compositions, fractions, angle=None):
        return clamped_layered_energy(compositions, fractions, self.substrate_lattice_parameter)


class ClampedLateral(ClampedLayered):
    """Columnar domains whose average in-plane lattice is clamped to the substrate."""

    def energy(self, compositions, fractions, angle=None):
        return clamped_lateral_energy(compositions, fractions, self.substrate_lattice_parameter)

### 4.2.5 Domains relaxed separately on a substrate (Hirata)

In [ ]:
class RelaxedClampedDomains(ElasticModel):
    """Domains clamped to the substrate independently, each keeping f(h, h_c(x)) of its clamping energy.

    Hirata's film model: misfit dislocations relax every domain separately; no strain between domains.
    film_thickness_nm=None: fully clamped (thinnest-film limit).
    """

    def __init__(self, substrate_lattice_parameter, film_thickness_nm, core_energy, core_radius):
        self.substrate_lattice_parameter = substrate_lattice_parameter
        self.relaxation = (film_thickness_nm, core_energy, core_radius)

    def energy(self, compositions, fractions, angle=None):
        energies = clamping_energies(compositions, self.substrate_lattice_parameter)
        if self.relaxation[0] is not None:
            energies = energies * relaxation_factors(compositions, *self.relaxation,
                                                     self.substrate_lattice_parameter)
        return np.sum(np.asarray(fractions) * energies, axis=-1)

### 4.2.6 Anisotropic and anharmonic elasticity — not implemented

Planned flavours: hexagonal $C_{ij}(x)$ (Zhang *et al.* 2013 give them up to $x = 0.375$) in place of the isotropic
moduli, and direct DFT epitaxial energies per interface orientation in place of linear elasticity.

## 4.3 Decomposition and stability at one composition

**Decomposition (binodal).** The film stays homogeneous or splits into two domains $x_B < \bar x < x_C$ with fraction
$f = (x_C-\bar x)/(x_C-x_B)$ of $x_B$:
$\Delta G_\mathrm{decomp} = [G(\bar x) + E_\mathrm{el}(\bar x)] - \min_{x_B,x_C}[f G(x_B) + (1-f) G(x_C) + E_\mathrm{el}(x_B, x_C, f)]$.
$\Delta G_\mathrm{decomp} > 0$: the film decomposes; the binodal is where it becomes zero. For incoherent domains this is
the common tangent; for coherent ones it is not, because $E_\mathrm{el}$ is not linear in $f$. The variables are $x_B$,
$x_C$ and, for the laminate, $\theta$; a gradient method is started from the best pairs of a grid spaced finely near
$\bar x$, from well-separated pairs, and from a small split, which guarantees that a film inside the spinodal is
found to decompose.

**Stability (spinodal).** The same comparison for a split $\bar x \mp \delta$ at $f = 1/2$ gives an effective curvature
$G''_\mathrm{eff} = 2[\ldots]/\delta^2$; the film is unstable where it is negative. For the laminate it is Cahn's
$\partial^2G/\partial x^2 + 2\eta^2 Y V_m$ minimised over orientation. Two-domain states only.

### 4.3.1 Search

In [ ]:
@dataclasses.dataclass
class DecompositionResult:
    """Best two-domain state at one average composition; energies in J/mol-f.u."""
    average_composition: float
    driving_force: float                  # > 0: decomposition lowers G; 0: stays homogeneous
    domain_compositions: tuple            # (x_B, x_C); both equal to the average if homogeneous
    fraction_of_lower_domain: float       # f of x_B
    interface_angle_degrees: float = None  # coherent laminate only; from the c axis
    geometry: str = None                   # coherent clamped only: "layered" or "lateral"


def pair_arrays(lower, upper, average):
    """Domain compositions and lever-rule fractions for arrays of pairs: shapes (..., 2)."""
    lower, upper = np.asarray(lower, dtype=float), np.asarray(upper, dtype=float)
    fraction = (upper - average) / (upper - lower)
    return np.stack([lower, upper], axis=-1), np.stack([fraction, 1.0 - fraction], axis=-1)


def chemical_part(gibbs_energy, temperature, compositions, fractions):
    """Fraction-weighted G of the domains, J/mol-f.u."""
    return np.sum(fractions * gibbs_energy(compositions, temperature), axis=-1)


def homogeneous_free_energy(gibbs_energy, temperature, average, elastic_model):
    """G + E_el of the undecomposed film at the average composition, J/mol-f.u."""
    elastic = elastic_model.minimum_energy(np.array([[average]]), np.array([[1.0]]))
    return float(gibbs_energy(np.array(average), temperature)) + float(np.squeeze(elastic))


def pair_free_energy(variables, gibbs_energy, temperature, average, elastic_model):
    """f G(x_B) + (1-f) G(x_C) + E_el for variables (x_B, x_C[, angle]) at one average composition."""
    compositions, fractions = pair_arrays(variables[0], variables[1], average)
    elastic = elastic_model.energy(compositions[None], fractions[None], *variables[2:])
    return float(chemical_part(gibbs_energy, temperature, compositions, fractions) + elastic[0])


def partner_axis(average, end):
    """Grid of partner compositions between the average and an end (0 or 1), denser near the average."""
    steps = np.linspace(1.0, 0.0, partner_grid_points, endpoint=False) ** 2
    axis = average + (end - average) * steps
    small = average + np.sign(end - average) * min(small_split, 0.5 * abs(end - average))
    return np.clip(np.append(axis, small), composition_margin, 1.0 - composition_margin)


def starting_points(gibbs_energy, temperature, average, elastic_model):
    """Starting points (x_B, x_C[, angle]): best grid pairs overall, best well-separated pairs, small split."""
    lower, upper = (axis.ravel() for axis in np.meshgrid(partner_axis(average, 0.0), partner_axis(average, 1.0),
                                                       indexing="ij"))
    compositions, fractions = pair_arrays(lower, upper, average)
    elastic, angles = elastic_model.energy_and_best_angle(compositions, fractions)
    order = np.argsort(chemical_part(gibbs_energy, temperature, compositions, fractions) + elastic)
    separated = order[upper[order] - lower[order] > separation]
    chosen = list(order[:refinement_starts]) + list(separated[:refinement_starts]) + [len(lower) - 1]
    return [(lower[i], upper[i]) + (() if angles is None else (angles[i],)) for i in dict.fromkeys(chosen)]


def variable_bounds(average, has_angle):
    """Bounds keeping x_B < average < x_C, and the angle within [0, 90 degrees]."""
    bounds = [(composition_margin, average - composition_margin), (average + composition_margin,
                                                                   1.0 - composition_margin)]
    return bounds + ([(0.0, 0.5 * math.pi)] if has_angle else [])


def decomposition(gibbs_energy, temperature, average_composition, elastic_model, method=refinement_method):
    """Best two-domain state of a film of average composition x (0 < x < 1); returns DecompositionResult.

    gibbs_energy(x, T): J/mol-f.u., vectorised in x. elastic_model: a section-17 ElasticModel.
    """
    average = float(average_composition)
    homogeneous = homogeneous_free_energy(gibbs_energy, temperature, average, elastic_model)
    best = None
    for start in starting_points(gibbs_energy, temperature, average, elastic_model):
        result = minimize(pair_free_energy, start, args=(gibbs_energy, temperature, average, elastic_model),
                          method=method, bounds=variable_bounds(average, len(start) == 3))
        best = result if best is None or result.fun < best.fun else best
    if best.fun >= homogeneous:
        return DecompositionResult(average, 0.0, (average, average), 1.0)
    lower, upper = best.x[:2]
    angle = math.degrees(best.x[2]) if len(best.x) == 3 else None
    return DecompositionResult(average, homogeneous - best.fun, (lower, upper),
                               (upper - average) / (upper - lower), angle)


def stability(gibbs_energy, temperature, average_composition, elastic_model,
              amplitude=fluctuation_amplitude):
    """Effective curvature (J/mol-f.u.) against a small +-amplitude two-domain split; < 0 is unstable."""
    average = float(average_composition)
    amplitude = min(amplitude, 0.5 * average, 0.5 * (1.0 - average))
    compositions, fractions = pair_arrays([average - amplitude], [average + amplitude], average)
    split = (chemical_part(gibbs_energy, temperature, compositions, fractions)[0]
             + float(np.squeeze(elastic_model.minimum_energy(compositions, fractions))))
    homogeneous = homogeneous_free_energy(gibbs_energy, temperature, average, elastic_model)
    return 2.0 * (split - homogeneous) / amplitude ** 2

### 4.3.2 Functions per case

| case | decomposition (binodal) | stability (spinodal) |
|---|---|---|
| free-standing, incoherent | `incoherent_decomposition` | `incoherent_stability` |
| free-standing, coherent | `coherent_decomposition` | `coherent_stability` |
| on a substrate, domains relaxed separately (Hirata) | `incoherent_clamped_decomposition` | `incoherent_clamped_stability` |
| on a substrate, coherent domains | `coherent_clamped_decomposition` | `coherent_clamped_stability` |

All take `gibbs_energy(x, T)`, `T` and $\bar x$; the clamped ones also the substrate lattice parameter (Å). Coherent
clamped films take `geometry = "layered"`, `"lateral"` or `"lowest"`. For a fully clamped film the layered geometry
and Hirata's case coincide.

In [ ]:
CLAMPED_GEOMETRIES = {"layered": ClampedLayered, "lateral": ClampedLateral}


def incoherent_decomposition(gibbs_energy, temperature, average_composition):
    """Free-standing, dislocated domains, each at its own lattice; returns DecompositionResult."""
    return decomposition(gibbs_energy, temperature, average_composition, IncoherentDomains())


def coherent_decomposition(gibbs_energy, temperature, average_composition):
    """Free-standing coherent domains, softest interface orientation; returns DecompositionResult."""
    return decomposition(gibbs_energy, temperature, average_composition, CoherentLaminate())


def incoherent_clamped_decomposition(gibbs_energy, temperature, average_composition, substrate_lattice_parameter,
                                     film_thickness_nm, core_energy, core_radius):
    """Film on a substrate, domains relaxed separately (Hirata); film_thickness_nm=None: fully clamped."""
    model = RelaxedClampedDomains(substrate_lattice_parameter, film_thickness_nm, core_energy, core_radius)
    return decomposition(gibbs_energy, temperature, average_composition, model)


def coherent_clamped_decomposition(gibbs_energy, temperature, average_composition, substrate_lattice_parameter,
                                   geometry="lowest"):
    """Coherent domains in a film clamped to a substrate lattice parameter (Angstrom); DecompositionResult."""
    if geometry != "lowest":
        result = decomposition(gibbs_energy, temperature, average_composition,
                               CLAMPED_GEOMETRIES[geometry](substrate_lattice_parameter))
        return dataclasses.replace(result, geometry=geometry if result.driving_force > 0.0 else None)
    return max((coherent_clamped_decomposition(gibbs_energy, temperature, average_composition,
                                               substrate_lattice_parameter, option)
                for option in CLAMPED_GEOMETRIES), key=operator.attrgetter("driving_force"))


def incoherent_stability(gibbs_energy, temperature, average_composition):
    """d2G/dx2, J/mol-f.u.; < 0 inside the chemical spinodal."""
    return stability(gibbs_energy, temperature, average_composition, IncoherentDomains())


def coherent_stability(gibbs_energy, temperature, average_composition):
    """Cahn coherent-spinodal curvature, softest orientation, J/mol-f.u."""
    return stability(gibbs_energy, temperature, average_composition, CoherentLaminate())


def incoherent_clamped_stability(gibbs_energy, temperature, average_composition, substrate_lattice_parameter,
                                 film_thickness_nm, core_energy, core_radius):
    """Spinodal curvature of a film whose domains relax separately (Hirata), J/mol-f.u."""
    model = RelaxedClampedDomains(substrate_lattice_parameter, film_thickness_nm, core_energy, core_radius)
    return stability(gibbs_energy, temperature, average_composition, model)


def coherent_clamped_stability(gibbs_energy, temperature, average_composition, substrate_lattice_parameter,
                               geometry="lowest"):
    """Spinodal curvature of a clamped coherent film, J/mol-f.u.; 'lowest' takes the less stable geometry."""
    if geometry != "lowest":
        return stability(gibbs_energy, temperature, average_composition,
                         CLAMPED_GEOMETRIES[geometry](substrate_lattice_parameter))
    return min(coherent_clamped_stability(gibbs_energy, temperature, average_composition,
                                          substrate_lattice_parameter, option) for option in CLAMPED_GEOMETRIES)

## 4.4 Phase boundaries and critical temperatures

Binodal: the driving force on a coarse scan of $\bar x$, every crossing of the threshold located by root-finding.
Spinodal: $G''_\mathrm{eff}$ on a grid. The substrate argument may be a number or a function of $\bar x$ (graded
substrate, or a partly relaxed coherent film).

In [ ]:
def substrate_at(substrate_lattice_parameter, composition):
    """Substrate lattice parameter for one average composition: a number or a function of x."""
    if callable(substrate_lattice_parameter):
        return substrate_lattice_parameter(composition)
    return substrate_lattice_parameter


def driving_force_at(composition, function, gibbs_energy, temperature, substrate_lattice_parameter=None):
    """Driving force (J/mol-f.u.) of one decomposition function at one average composition."""
    arguments = (gibbs_energy, temperature, composition)
    if substrate_lattice_parameter is not None:
        arguments += (substrate_at(substrate_lattice_parameter, composition),)
    return function(*arguments).driving_force


def is_decomposing(composition, stability_function, function, gibbs_energy, temperature,
                   substrate_lattice_parameter=None):
    """True if the film decomposes; the cheap spinodal test is tried first when given."""
    if stability_function is not None:
        arguments = (gibbs_energy, temperature, composition)
        if substrate_lattice_parameter is not None:
            arguments += (substrate_at(substrate_lattice_parameter, composition),)
        if stability_function(*arguments) < 0.0:
            return True
    return threshold_excess(composition, function, gibbs_energy, temperature, substrate_lattice_parameter) > 0.0


def threshold_excess(composition, function, gibbs_energy, temperature, substrate_lattice_parameter=None):
    return driving_force_at(composition, function, gibbs_energy, temperature,
                            substrate_lattice_parameter) - decomposition_threshold


def binodal_compositions(function, gibbs_energy, temperature, substrate_lattice_parameter=None,
                         stability_function=None, scan=binodal_scan):
    """Average compositions at which the driving force crosses the threshold, at one temperature.

    stability_function (optional): the matching spinodal test; a spinodal-unstable film is inside the binodal,
    which skips its decomposition search.
    """
    arguments = (function, gibbs_energy, temperature, substrate_lattice_parameter)
    inside = np.array([is_decomposing(x, stability_function, *arguments) for x in scan])
    crossings = np.flatnonzero(inside[1:] != inside[:-1])
    return [brentq(threshold_excess, scan[i], scan[i + 1], args=arguments, xtol=1e-4) for i in crossings]


def phase_map(function, gibbs_energy, temperatures, compositions, substrate_lattice_parameter=None, **options):
    """function(gibbs_energy, T, x[, substrate], **options) on a (T, x) grid -> array (len(T), len(x)).

    Decomposition functions contribute their driving force; stability functions their curvature.
    """
    values = np.zeros((len(temperatures), len(compositions)))
    for row, temperature in enumerate(temperatures):
        for column, composition in enumerate(compositions):
            arguments = (gibbs_energy, temperature, composition)
            if substrate_lattice_parameter is not None:
                arguments += (substrate_at(substrate_lattice_parameter, composition),)
            result = function(*arguments, **options)
            values[row, column] = getattr(result, "driving_force", result)
    return values


def critical_temperature(stability_function, gibbs_energy, compositions, substrate_lattice_parameter=None,
                         low=100.0, high=9000.0, iterations=16):
    """Highest T at which the stability function is negative somewhere on the composition grid; (T, x).

    (nan, nan): stable at `low`; (inf, nan): still unstable at `high`.
    """
    if not np.any(phase_map(stability_function, gibbs_energy, [low], compositions,
                            substrate_lattice_parameter) < 0.0):
        return float("nan"), float("nan")
    if np.any(phase_map(stability_function, gibbs_energy, [high], compositions,
                        substrate_lattice_parameter) < 0.0):
        return float("inf"), float("nan")
    for _ in range(iterations):
        middle = 0.5 * (low + high)
        unstable = np.any(phase_map(stability_function, gibbs_energy, [middle], compositions,
                                    substrate_lattice_parameter) < 0.0)
        low, high = (middle, high) if unstable else (low, middle)
    curvature = phase_map(stability_function, gibbs_energy, [low], compositions, substrate_lattice_parameter)[0]
    return low, float(compositions[int(np.argmin(curvature))])

## 4.5 Substrate strain as an interaction-parameter shift

Hirata's clamped energy fitted on the buckled branch to a line plus $x(1-x)[\Delta L^{(0)} + \Delta L^{(1)}(1-2x)]$. A
near-exact fit means a strain term and a change of $L^{(0)}$ cannot be told apart from one measured boundary.

In [ ]:
strain_fit_compositions = np.linspace(0.0, 0.45, 46)    # buckled branch, film compositions


def substrate_strain_energy(composition, substrate_lattice_parameter, film_thickness_nm=None):
    """Hirata's clamped-film energy with relaxation factor f(h, h_c(x)) (None: fully clamped), J/mol-f.u."""
    compositions = np.atleast_1d(np.asarray(composition, dtype=float))
    energy = clamping_energies(compositions[..., None], substrate_lattice_parameter)[..., 0]
    if film_thickness_nm is not None:
        energy = energy * relaxation_factors(compositions, film_thickness_nm, chosen_core_energy, chosen_core_radius,
                                             substrate_lattice_parameter)
    return energy


def strain_as_interaction_parameters(substrate_lattice_parameter, film_thickness_nm):
    """(dL0, dL1) in J/mol-f.u. and the relative rms residual of the fit, 0 <= x <= 0.45."""
    x = strain_fit_compositions
    energies = substrate_strain_energy(x, substrate_lattice_parameter, film_thickness_nm)
    basis = np.stack([np.ones_like(x), x, x * (1.0 - x), x * (1.0 - x) * (1.0 - 2.0 * x)], axis=1)
    coefficients, *_ = np.linalg.lstsq(basis, energies, rcond=None)
    excess = energies - basis[:, :2] @ coefficients[:2]
    residual = energies - basis @ coefficients
    return coefficients[2:], math.sqrt(np.mean(residual ** 2)) / np.max(np.abs(excess))

## 4.6 Interface for the defect model

The free energy of the **homogeneous** film, relative to the unstrained AlN–ScN wurtzite endmembers, J/mol-f.u.; with a
substrate it includes the biaxial clamping energy.

In [ ]:
def homogeneous_film_gibbs_energy(gibbs_energy, temperature, composition, substrate_lattice_parameter=None):
    """G of an undecomposed film; substrate_lattice_parameter: None (free) or Angstrom."""
    composition = np.asarray(composition, dtype=float)
    energy = gibbs_energy(composition, temperature)
    if substrate_lattice_parameter is None:
        return energy
    return energy + substrate_strain_energy(composition, substrate_lattice_parameter).reshape(np.shape(energy))

# 5. Auxiliary: cases and plotting helpers

## 5.1 Cases

In [ ]:
def with_thickness(function, film_thickness_nm):
    """An incoherent-clamped function with thickness and core parameters bound."""
    return functools.partial(function, film_thickness_nm=film_thickness_nm, core_energy=chosen_core_energy,
                             core_radius=chosen_core_radius)


relaxed_aluminium_nitride = functools.partial(partially_relaxed_lattice, substrate_lattice_parameter=aluminium_nitride,
                                              film_thickness_nm=real_film_thickness_nm,
                                              core_energy=chosen_core_energy, core_radius=chosen_core_radius)
# name: (decomposition function, stability function, substrate: None, number, or function of x)
cases = {
    "free-standing, incoherent": (incoherent_decomposition, incoherent_stability, None),
    "free-standing, coherent": (coherent_decomposition, coherent_stability, None),
    "fully clamped on AlN, incoherent": (with_thickness(incoherent_clamped_decomposition, None),
                                         with_thickness(incoherent_clamped_stability, None), aluminium_nitride),
    "fully clamped on AlN, coherent": (coherent_clamped_decomposition, coherent_clamped_stability, aluminium_nitride),
    "1 um on AlN, incoherent": (with_thickness(incoherent_clamped_decomposition, real_film_thickness_nm),
                                with_thickness(incoherent_clamped_stability, real_film_thickness_nm), aluminium_nitride),
    "1 um on AlN, coherent": (coherent_clamped_decomposition, coherent_clamped_stability, relaxed_aluminium_nitride),
}

## 5.2 Computing and drawing

In [ ]:
def decomposition_at(name, temperature, composition):
    decompose, _, substrate = cases[name]
    arguments = (wurtzite_gibbs_energy, temperature, composition)
    return decompose(*arguments + (() if substrate is None else (substrate_at(substrate, composition),)))


def describe(result):
    """'driving force (x_B + x_C) geometry' in meV/f.u."""
    lower, upper = result.domain_compositions
    geometry = (f"{result.interface_angle_degrees:3.0f} deg" if result.interface_angle_degrees is not None
                else result.geometry or "")
    return f"{result.driving_force * MILLIELECTRONVOLT_PER_JOULE_PER_MOLE:6.1f} ({lower:.3f}+{upper:.3f}) {geometry}"


@functools.lru_cache(maxsize=None)
def phase_boundaries(name):
    """Binodal points [(T, x)] and spinodal curvature map of one case; cached, as each takes minutes."""
    decompose, stable, substrate = cases[name]
    binodal = [(temperature, x) for temperature in binodal_temperatures
               for x in binodal_compositions(decompose, wurtzite_gibbs_energy, temperature, substrate, stable)]
    spinodal = phase_map(stable, wurtzite_gibbs_energy, spinodal_temperatures, spinodal_compositions, substrate)
    return binodal, spinodal


def conditions(panel):
    """One line stating the assumptions behind a panel's results."""
    film = {"free-standing": "free-standing (bulk, or film fully relaxed)",
            "fully clamped on AlN": "film fully clamped to AlN (h < h_c)",
            "1 um on AlN": f"{real_film_thickness_nm:.0f} nm film on AlN, h_c with {chosen_core_energy.__name__}, "
                           f"{chosen_core_radius.__name__}"}[panel]
    corrections = " + ".join(chemical_corrections) or "none"
    return f"Conditions: {film}; Hirata G, chemical corrections: {corrections}; transition width {lattice_transition_width}"


def draw_boundaries(axis, name, colour, label):
    """Binodal points and spinodal contour (dotted) of one case."""
    binodal, spinodal = phase_boundaries(name)
    axis.plot([x for _, x in binodal], [temperature for temperature, _ in binodal], "o", color=colour,
              markersize=2.5, label=f"{label}, binodal")
    axis.contour(spinodal_compositions, spinodal_temperatures, spinodal, levels=[0.0], colors=colour, linestyles=":")
    axis.plot([], [], ":", color=colour, label=f"{label}, spinodal")


def plot_phase_diagram(panel):
    """Incoherent and coherent binodals and spinodals of one panel, with the free-standing incoherent reference."""
    figure, axis = plt.subplots(figsize=(7.5, 5.5))
    if panel != "free-standing":
        draw_boundaries(axis, "free-standing, incoherent", "lightgrey", "free-standing incoherent (reference)")
    draw_boundaries(axis, f"{panel}, incoherent", "k", "incoherent")
    draw_boundaries(axis, f"{panel}, coherent", "tab:red", "coherent")
    axis.axhspan(growth_temperatures[0], growth_temperatures[-1], color="tab:olive", alpha=0.2)
    axis.axvline(TRANSITION_COMPOSITION, color="grey", ls="--", lw=0.8)
    axis.set_xlim(0, 1); axis.set_xlabel("Sc cation fraction $x$"); axis.set_ylabel("Temperature (K)")
    axis.set_title(f"{panel} (band: growth window 300-450 C; dashed: x0)")
    axis.legend(fontsize=7.5, loc="upper right"); axis.grid(alpha=0.25)
    print(conditions(panel))
    plt.tight_layout(); plt.show()


def plot_driving_force(panel, compositions=np.linspace(0.02, 0.98, 49)):
    """Decomposition driving force of one panel at the growth temperatures, meV/f.u."""
    figure, axis = plt.subplots(figsize=(7.5, 4.5))
    styles = dict(zip(growth_temperatures, (":", "-", "--")))
    for kind, colour in (("incoherent", "k"), ("coherent", "tab:red")):
        decompose, _, substrate = cases[f"{panel}, {kind}"]
        for temperature in growth_temperatures:
            values = phase_map(decompose, wurtzite_gibbs_energy, [temperature], compositions, substrate)[0]
            axis.plot(compositions, values * MILLIELECTRONVOLT_PER_JOULE_PER_MOLE, styles[temperature],
                      color=colour, label=f"{kind}, {temperature - 273.15:.0f} C")
    axis.axvline(TRANSITION_COMPOSITION, color="grey", ls="--", lw=0.8)
    axis.set_xlabel("Sc cation fraction $x$"); axis.set_ylabel(r"$\Delta G_\mathrm{decomp}$ (meV/f.u.)")
    axis.set_title(f"Decomposition driving force: {panel}"); axis.legend(fontsize=7.5); axis.grid(alpha=0.25)
    print(conditions(panel))
    plt.tight_layout(); plt.show()


def print_best_states(panel, compositions=(0.1, 0.2, 0.3, 0.4, 0.5)):
    """Driving force, domain compositions and geometry at the growth temperature."""
    print(f"Best state at {growth_temperature - 273.15:.0f} C: driving force (meV/f.u.), domains x_B+x_C, "
          "geometry (laminate: interface-normal angle from c)")
    for kind in ("incoherent", "coherent"):
        print(f"  {kind}")
        for composition in compositions:
            result = decomposition_at(f"{panel}, {kind}", growth_temperature, composition)
            print(f"    x = {composition:.1f}:  {describe(result)}")


def plot_critical_thickness(core_energies, core_radius, substrates=("AlN", "GaN"),
                            compositions=np.linspace(0.02, 0.5, 97)):
    """h_c(x) on the given substrates for several core-energy functions."""
    figure, axis = plt.subplots(figsize=(7.5, 4.5))
    for energy, style in zip(core_energies, ("-", "--", ":")):
        for substrate, colour in zip(substrates, ("tab:blue", "tab:green")):
            axis.plot(compositions, [critical_thickness_nm(x, energy, core_radius, substrate_in_plane_lattice[substrate])
                                     for x in compositions], style, color=colour,
                      label=f"on {substrate}, {energy.__name__.replace('dummy_core_energy_', '')}")
    axis.set_yscale("log"); axis.set_ylim(0.5, 1e3)
    axis.set_xlabel("Sc cation fraction $x$"); axis.set_ylabel("critical thickness $h_c$ (nm)")
    axis.legend(fontsize=7.5); axis.grid(alpha=0.25, which="both")
    print(f"Conditions: energy balance, isotropic elasticity; core radius {core_radius.__name__}; all core values DUMMY")
    plt.tight_layout(); plt.show()


def print_critical_temperatures(names):
    """Highest temperature with a spinodal instability, over all x and on the buckled branch."""
    buckled = critical_compositions[critical_compositions < TRANSITION_COMPOSITION - 0.05]
    print(f"{'':34s} {'all x':>22} {'x < x0 - 0.05':>22}")
    for name in names:
        _, stable, substrate = cases[name]
        results = [critical_temperature(stable, wurtzite_gibbs_energy, grid, substrate)
                   for grid in (critical_compositions, buckled)]
        print(f"  {name:32s} " + " ".join(
            f"{'no gap':>22}" if math.isnan(temperature) else f"{'unstable at all T':>22}" if math.isinf(temperature)
            else f"{temperature:8.0f} K at x = {composition:.3f}" for temperature, composition in results))


def print_interaction_parameter_shifts(substrates=("AlN", "GaN"), thicknesses_nm=(10.0, 100.0, 1000.0)):
    """Clamped strain energy as Redlich-Kister shifts, meV/f.u."""
    print(f"{'substrate':>9} {'h (nm)':>7} {'dL0':>8} {'dL1':>8} {'rel. rms':>9}   meV/f.u.; L0 = "
          f"{REDLICH_KISTER_WURTZITE[0] * MILLIELECTRONVOLT_PER_JOULE_PER_MOLE:.0f}")
    for substrate in substrates:
        for thickness_nm in thicknesses_nm:
            (delta_l0, delta_l1), relative = strain_as_interaction_parameters(
                substrate_in_plane_lattice[substrate], thickness_nm)
            print(f"{substrate:>9} {thickness_nm:7.0f} {delta_l0 * MILLIELECTRONVOLT_PER_JOULE_PER_MOLE:8.1f} "
                  f"{delta_l1 * MILLIELECTRONVOLT_PER_JOULE_PER_MOLE:8.1f} {relative:9.4f}")
    print(f"Conditions: Hirata film model, h_c with {chosen_core_energy.__name__}, {chosen_core_radius.__name__}")

# 6. Results

## 6.1 Critical thickness

In [ ]:
plot_critical_thickness((dummy_core_energy_high_constant, dummy_core_energy_low_constant, dummy_core_energy_high_slope),
                        dummy_core_radius)

## 6.2 Critical temperatures

In [ ]:
print_critical_temperatures(cases)

## 6.3 Phase diagrams

Points: binodal; dotted: spinodal. Each panel pairs the incoherent case with the coherent one; grey: free-standing
incoherent, for reference.

### 6.3.1 Free-standing

In [ ]:
plot_phase_diagram("free-standing")

### 6.3.2 Fully clamped on AlN

The thinnest-film limit. Narrow bands near $x_0$ that stay unstable at every temperature come from the steep $a(x)$
at the transition under ~10 % compression; their strength depends on the assumed transition width.

In [ ]:
plot_phase_diagram("fully clamped on AlN")

### 6.3.3 1 µm film on AlN

Above $x \approx 0.43$ the critical thickness reaches the core radius, where the energy-balance model does not apply.

In [ ]:
plot_phase_diagram("1 um on AlN")

## 6.4 Decomposition driving force at growth temperatures

### 6.4.1 Free-standing

In [ ]:
plot_driving_force("free-standing")
print_best_states("free-standing")

### 6.4.2 Fully clamped on AlN

In [ ]:
plot_driving_force("fully clamped on AlN")
print_best_states("fully clamped on AlN")

### 6.4.3 1 µm film on AlN

In [ ]:
plot_driving_force("1 um on AlN")
print_best_states("1 um on AlN")

## 6.5 Substrate strain as an interaction-parameter shift

If the relative rms is small, strain and $L^{(0)}$ are degenerate against a single measured boundary: separating them
needs boundaries at two or more thicknesses or substrates — a design requirement for Library 1.

In [ ]:
print_interaction_parameter_shifts()